In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os

In [ ]:
# =========================================================
# 1. PATH SETUP
# =========================================================
TRACT_SHAPEFILE_PATH  = "../data/raw/census/tl_2023_36_tract.shp"
NYC_WILD_CSV_PATH     = "../data/raw/forever_wild/NYC_Parks_Forever_Wild_20260205.csv"
PARK_PROPERTIES_PATH  = "../data/raw/park_properties/geo_export_d47a2214-9d99-4d6d-b889-0aa5a59bb182.shp"
ADVAN_PATH            = "../data/raw/advan"
ADVAN_FILES           = [os.path.join(ADVAN_PATH, f) for f in os.listdir(ADVAN_PATH) if f.endswith(".csv")]
FLOW_FILE             = "../data/intermediate/park_flows_placekey.csv"
OUTPUT_FILE           = "../data/output/parks_audit.csv"
CENTROID_OUTPUT       = "../data/intermediate/park_property_centroids.csv"
PROP_PLACEKEYS_OUTPUT = "../data/intermediate/property_placekeys.csv"


In [ ]:
# =========================================================
# 2. PREPARE GEOGRAPHY — NYC TRACTS
# =========================================================
print("Preparing NYC geography...")
tracts = gpd.read_file(TRACT_SHAPEFILE_PATH)
nyc_counties = ['005', '047', '061', '081', '085']

# Project to State Plane (feet) before unioning so containment tests below
# use linear units, not lat/lon degrees.
nyc_tracts = tracts[tracts['COUNTYFP'].isin(nyc_counties)].copy().to_crs("EPSG:2263")

# Single dissolved polygon for the NYC footprint, used to filter Advan POIs
nyc_union = nyc_tracts.geometry.union_all()

In [ ]:
# =========================================================
# 3. LOAD PARK PROPERTIES SHAPEFILE
# =========================================================
print("Loading NYC park properties shapefile...")

# gisobjid is the unique row id; gispropnum groups POIs that share a parent
# property (multi-piece parks). signname / acres come straight from NYC Parks.
parks_shp = gpd.read_file(PARK_PROPERTIES_PATH)[[
    'gisobjid', 'gispropnum', 'signname', 'acres', 'geometry'
]].to_crs("EPSG:2263")
print(f"Park properties loaded: {len(parks_shp)} polygons")

In [ ]:
# =========================================================
# 4. LOAD FOREVER WILD SITES
#
# A single property (gispropnum) can contain multiple
# Forever Wild sites. We keep one row per property:
#   - forever_wild_id  : first FW ID (used as boolean flag)
#   - fw_acres         : sum of all FW site acres at that property
# =========================================================
print("Loading Forever Wild sites...")

# Note: 'ForverWildID' is a typo in the source CSV header; we mirror it here
# rather than fight it. Strip whitespace from headers defensively.
fw_raw = (pd.read_csv(NYC_WILD_CSV_PATH)
          .rename(columns=lambda c: c.strip())
          [['GISPropNum', 'ForverWildID', 'Acres']]
          .rename(columns={
              'GISPropNum':   'gispropnum',
              'ForverWildID': 'forever_wild_id',
              'Acres':        'fw_acres',
          }))

# The CSV's `shape` WKT field can throw off pandas' dtype inference and
# leave Acres parsed as object/string. Coerce to float; bad cells -> NaN.
fw_raw['fw_acres'] = pd.to_numeric(fw_raw['fw_acres'], errors='coerce')

# Collapse multiple FW sites per property into a single row by summing acres
fw = (fw_raw.groupby('gispropnum', as_index=False)
            .agg(forever_wild_id=('forever_wild_id', 'first'),
                 fw_acres=('fw_acres', 'sum')))
print(f"Forever Wild properties loaded: {len(fw)}")
fw.head()

In [ ]:
# =========================================================
# 5. COLLECT UNIQUE PARKS + TOTAL VISITS FROM ADVAN
# =========================================================
print("Extracting parks and aggregating visits from Advan files...")
parks_list  = []
visits_list = []

# Each monthly file repeats the same POIs but with different visit counts.
# Build two lists: one keeps each POI's static metadata, the other keeps
# its visit counts so we can sum across months.
for file_path in ADVAN_FILES:
    df = pd.read_csv(file_path, usecols=[
        'PLACEKEY', 'PARENT_PLACEKEY', 'LOCATION_NAME',
        'LATITUDE', 'LONGITUDE', 'RAW_VISIT_COUNTS'
    ])
    parks_list.append(
        df[['PLACEKEY', 'PARENT_PLACEKEY', 'LOCATION_NAME', 'LATITUDE', 'LONGITUDE']]
        .drop_duplicates(subset=['PLACEKEY'])
    )
    visits_list.append(df[['PLACEKEY', 'RAW_VISIT_COUNTS']])

# Deduplicate metadata (any month's row is fine — it's static per placekey)
parks_combined  = pd.concat(parks_list).drop_duplicates(subset=['PLACEKEY'])

# Sum visit counts across the three months per POI
visits_combined = (pd.concat(visits_list)
                   .groupby('PLACEKEY', as_index=False)['RAW_VISIT_COUNTS'].sum()
                   .rename(columns={'RAW_VISIT_COUNTS': 'total_visits'}))
parks_combined = parks_combined.merge(visits_combined, on='PLACEKEY', how='left')
print(f"Unique parks found: {len(parks_combined)}")
parks_combined.head()

In [ ]:
# =========================================================
# 5b. FILTER PARKS — within NYC boundary or has NYC flows
# =========================================================
print("Filtering parks to NYC...")

# Build a GeoDataFrame from the lat/lon columns in WGS84, then project for
# accurate within() tests against the NYC boundary.
geometry       = [Point(xy) for xy in zip(parks_combined['LONGITUDE'], parks_combined['LATITUDE'])]
gdf_parks      = gpd.GeoDataFrame(parks_combined, geometry=geometry, crs="EPSG:4326")
gdf_parks_proj = gdf_parks.to_crs("EPSG:2263")

# Two-prong keep rule: either the POI sits inside NYC, or NYC residents
# visited it (it shows up in the flow file). The second prong catches
# cross-boundary parks — e.g. parks just over the border that NYC residents
# visit — without admitting random non-NYC POIs.
in_nyc                = gdf_parks_proj.geometry.within(nyc_union)
flow_df               = pd.read_csv(FLOW_FILE)
parks_with_nyc_visits = set(flow_df['park_j'].unique())
keep_mask             = in_nyc.values | gdf_parks['PLACEKEY'].isin(parks_with_nyc_visits)

gdf_parks      = gdf_parks[keep_mask].reset_index(drop=True)
gdf_parks_proj = gdf_parks_proj[keep_mask].reset_index(drop=True)
print(f"Parks after filtering: {len(gdf_parks)}")

In [ ]:
# =========================================================
# 6. SPATIAL JOIN — Advan points → Park Properties polygons
# =========================================================
print("Joining Advan park points to NYC park property polygons...")

# Primary attempt: each POI must fall WITHIN a park polygon
joined_properties = gpd.sjoin(
    gdf_parks_proj[['PLACEKEY', 'geometry']],
    parks_shp,
    how='left',
    predicate='within'
)

# A handful of POIs sit just outside the polygon (Advan GPS drift, mismatched
# polygon edges). Re-run those as nearest-neighbour with no distance cap and
# patch them into the result.
unmatched_mask = joined_properties['gisobjid'].isna()
if unmatched_mask.any():
    unmatched_pts = gdf_parks_proj[['PLACEKEY', 'geometry']][
        gdf_parks_proj['PLACEKEY'].isin(joined_properties[unmatched_mask]['PLACEKEY'])
    ]
    nearest = gpd.sjoin_nearest(
        unmatched_pts, parks_shp, how='left'
    ).drop_duplicates(subset='PLACEKEY')

    # update() fills NaN cells in the within() result with values from
    # nearest. Indexing both DataFrames by PLACEKEY makes the alignment safe.
    joined_properties = joined_properties.drop_duplicates(subset='PLACEKEY').set_index('PLACEKEY')
    nearest = nearest.set_index('PLACEKEY')
    joined_properties.update(nearest)
    joined_properties = joined_properties.reset_index()

joined_properties = joined_properties.drop_duplicates(subset='PLACEKEY')

matched = joined_properties['gisobjid'].notna().sum()
print(f"Matched to park polygon: {matched}/{len(joined_properties)} ({matched/len(joined_properties):.1%})")

In [ ]:
# =========================================================
# 7. BUILD FINAL AUDIT TABLE
# =========================================================
print("Building audit output...")

# Start from the per-POI metadata + summed visit counts from step 5
audit = gdf_parks[['PLACEKEY', 'PARENT_PLACEKEY', 'LOCATION_NAME', 'total_visits']].copy()

# Attach gispropnum, signname and acres from the park properties spatial join
audit = audit.merge(
    joined_properties[['PLACEKEY', 'gispropnum', 'signname', 'acres']],
    on='PLACEKEY',
    how='left'
)

# Attach forever_wild_id and fw_acres via gispropnum lookup. how='left'
# leaves non-FW properties with NaN for both columns.
audit = audit.merge(fw, on='gispropnum', how='left')

# nature_fraction: share of property area that is Forever Wild,
# clipped to [0, 1]. NaN where the property has no acres reported.
# Properties with no FW site get fw_acres = 0 → nature_fraction = 0.
audit['fw_acres'] = audit['fw_acres'].fillna(0)
audit['nature_fraction'] = (audit['fw_acres'] / audit['acres']).clip(lower=0, upper=1)

# Properties with missing/zero acres can't yield a meaningful fraction.
# Mark them NaN so downstream code can decide whether to drop or impute.
audit.loc[audit['acres'].isna() | (audit['acres'] <= 0), 'nature_fraction'] = np.nan

# Final rename and column selection
audit = audit.rename(columns={
    'PLACEKEY':        'placekey',
    'PARENT_PLACEKEY': 'parent_placekey',
    'LOCATION_NAME':   'name',
    'gispropnum':      'gis_prop_num',
    'signname':        'property_name',
    'total_visits':    'visits',
})

audit = audit[[
    'placekey',
    'parent_placekey',
    'name',
    'gis_prop_num',
    'property_name',
    'visits',
    'acres',
    'forever_wild_id',
    'nature_fraction',
]]

audit = audit.sort_values(['gis_prop_num', 'placekey']).reset_index(drop=True)
audit.to_csv(OUTPUT_FILE, index=False)
print(f"\nAudit written to: {OUTPUT_FILE}")
print(f"Total parks:        {len(audit)}")
print(f"Forever Wild parks: {audit['forever_wild_id'].notna().sum()}")
print(f"nature_fraction summary:")
print(audit['nature_fraction'].describe())
audit.head()

In [ ]:
# =========================================================
# 8. SAVE PROPERTY CENTROID LOOKUP
# =========================================================

# One row per gispropnum. parks_shp may have multiple rows per property
# (multi-piece parks); drop_duplicates picks the first polygon for each.
# Centroids are still in EPSG:2263 (feet) — distance calc happens in 05.
property_centroids = (
    parks_shp[['gispropnum', 'geometry']]
    .drop_duplicates(subset='gispropnum')
    .copy()
)
property_centroids['centroid_x'] = property_centroids.geometry.centroid.x
property_centroids['centroid_y'] = property_centroids.geometry.centroid.y

property_centroids = property_centroids[['gispropnum', 'centroid_x', 'centroid_y']].rename(
    columns={'gispropnum': 'gis_prop_num'}
)

property_centroids.to_csv(CENTROID_OUTPUT, index=False)
print(f"Property centroids saved: {len(property_centroids)}")
property_centroids.head()

In [ ]:
# =========================================================
# 9. SAVE PROPERTY → PLACEKEYS LOOKUP
# Built from audit table — includes ALL POIs per property,
# even those with no flow records
# =========================================================

# Why source from the audit (not from flow_df)? The audit covers every
# Advan POI in NYC, including those with zero flows. This means downstream
# property aggregations can list the full POI roster rather than only the
# subset that happened to have visits in this 3-month window.
prop_placekeys = (
    audit[['placekey', 'gis_prop_num']]
    .dropna(subset=['gis_prop_num'])
    .groupby('gis_prop_num')['placekey']
    .apply(lambda x: list(x.unique()))
    .reset_index()
    .rename(columns={'placekey': 'all_placekeys'})
)

prop_placekeys.to_csv(PROP_PLACEKEYS_OUTPUT, index=False)
print(f"Property-placekey lookup saved: {len(prop_placekeys)} properties")
prop_placekeys.head()